# NSCLC Therapy Agent — Evaluation Framework Demo

This notebook demonstrates the **modularized** `nsclc_eval` package end-to-end on synthetic demo data.

It uses a **deterministic mock LLM client** so the whole pipeline runs offline, without a GPU or a `llama-cpp-server`. To run against real open-weight models, see the "Run with real models" section at the bottom.

> **Privacy note:** the demo data in `data/sample/` is fully synthetic. Real patient data is never stored in this repository.

## 0. Imports

Make the `src/` package importable without installing it.

In [ ]:
from nsclc_eval.pipeline import run_all_patient_evaluations, DEFAULT_EVAL_FLAGS
from nsclc_eval.data_loading import load_ground_truth
from nsclc_eval.testing import MockLLMClient
from nsclc_eval import config
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))


print(f"Repo root: {REPO_ROOT}")

## 1. Configure an offline mock client

Inject the deterministic mock LLM and point the package at the synthetic data.

In [ ]:
# Offline mode: use the deterministic mock instead of a live llama-cpp-server
config.configure(
    client_instance=MockLLMClient(),
    judge_models=["mock-judge-1", "mock-judge-2"],
    seed=42,
    temperature=0.2,
)

config.DATA_DIR = str(REPO_ROOT / "data/sample")
config.GROUND_TRUTH_CSV = str(
    REPO_ROOT / "data/sample/sample_ground_truth.csv")

print(f"Judges: {config.JUDGE_MODELS}")

## 2. Load ground truth

In [ ]:
gt = load_ground_truth(config.GROUND_TRUTH_CSV)
print(f"Ground truth loaded: {len(gt)} patients")
gt

## 3. Run the full evaluation matrix

The pipeline evaluates every patient JSON with every judge model, computing:

* **Tier 1 Core** — Reasoning Efficiency Score (RES)
* **Tier 1 Extended** — Reasoning Completeness, Guideline Adherence (GAR), Factuality, Faithfulness
* **Tier 2** — Binary accuracy (IO vs IOCT), Treatment Plan Completeness

`restart_server=False` skips the Docker VRAM management (needed only with a real local server).

In [ ]:
results = run_all_patient_evaluations(
    data_dir=config.DATA_DIR,
    gt_mapping=gt,
    judge_models=config.JUDGE_MODELS,
    mode="both",
    orchestrators_to_evaluate=None,
    max_files_per_batch=None,
    save_results=True,
    verbose=True,
    eval_flags=DEFAULT_EVAL_FLAGS,
    restart_server=False,
)

## 4. Summary of evaluations

In [ ]:
import pandas as pd

summary = pd.DataFrame(results)
cols = [
    "patient_id", "orchestrator_model", "judge_model",
    "res_score", "binary_accuracy", "gar_score",
    "reasoning_completeness_score", "treatment_completeness_score",
    "factuality_score", "faithfulness_score",
]
summary[cols].sort_values(
    ["orchestrator_model", "judge_model"]).reset_index(drop=True)

## 5. Run with real models (on your GPU machine)

To run the real evaluation on a machine with the Docker `llama-cpp-server` (CUDA):

```python
from nsclc_eval import config
from nsclc_eval.data_loading import load_ground_truth
from nsclc_eval.pipeline import run_all_patient_evaluations

# Point at your real patient JSON directory and ground-truth CSV
config.DATA_DIR = "data/your_real_patients"
config.GROUND_TRUTH_CSV = "data/your_ground_truth.csv"
gt = load_ground_truth(config.GROUND_TRUTH_CSV)

results = run_all_patient_evaluations(
    data_dir=config.DATA_DIR,
    gt_mapping=gt,
    judge_models=config.JUDGE_MODELS,   # loaded from config/models.ini
    restart_server=True,                # restart Docker on judge swap (VRAM flush)
)
```

Outputs are written to `eval/<date>/<time>/<judge>/`.